In [ ]:
from bpcagent import BasicAgent, CalculatorTool, Config, ToolRegistry

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

project_dir = Path("/Users/elsie/Documents/个人/Github/TargetAgent/BPCAgent")
load_dotenv(project_dir / ".env", override=False)

config = Config()

registry = ToolRegistry()
calculator = CalculatorTool()
registry.register_tool(calculator)

agent = BasicAgent(
    name="Assistant",
    config=config,
    sys_prompt="你是一个简洁的助手，计算问题请调用工具。",
    tool_registry=registry,
)
# 包装实际方法，只验证执行次数，仍运行计算器原始代码。
# from unittest.mock import patch
# with patch.object(calculator, "run", wraps=calculator.run) as execution:
#     answer = agent.run("请调用计算器计算 137+55*123。")
#     assert execution.call_count > 0, "模型未实际调用计算器"
#     print(f"计算器实际执行了 {execution.call_count} 次")

result = agent.run("请调用计算器计算 137+55*123。")
if result.finished is False:
    print(result.info)
else:
    # print(result)
    print(result.output.get("text", ""))


## 完整 AgentWorkflow：DeepSeek 真实调用
这一节可以独立执行，不依赖上面的旧单 Agent 示例。请使用安装了本仓库的 Python kernel。

PatientState → Review → TargetSelection → Review → DrugSelection → Review；Review 要求修改时会重跑当前阶段。Meta 暂不运行。三个任务使用 DeepSeek Flash，Review 使用 DeepSeek V4 Pro。

**运行 live 会消耗 API 额度，并将截止日内的公开患者数据发送到 DeepSeek。** 密钥只从 config/secrets.local.json 或环境变量读取，不在 notebook 中填写或打印。改成 demo 可以离线验证。科学工具仍为占位，跑通不等于产生有效的临床药物推荐。

In [ ]:
from pathlib import Path
from bpcagent.workflow import AgentWorkflow
from bpcagent.workflow.config import WorkflowConfiguration

PROJECT_DIR = Path('/home/u2021202188/BPCAgent')
MODE = 'live'  # 可选：'live'（真实 API）或 'demo'（离线）
PATIENT_ID = 'GENIE-DFCI-051542'
CUTOFF_DAY = 50  # 先用较短历史跑通；可改为 249 等截止日
START_DAY = None  # 可选：近期观察窗口的起始相对日，不丢弃更早的治疗历史
configuration = WorkflowConfiguration(PROJECT_DIR / 'config/workflow.jsonc')
options = configuration.workflow_options({'mode': MODE})
model_options = configuration.model_configs() if MODE == 'live' else {}
workflow = AgentWorkflow(**options, **model_options)
{role: {'provider': cfg.default_provider, 'model': cfg.model}
 for role, cfg in workflow.configs.items()}  # 不展示 cfg 全对象或 API key


In [ ]:
import os
from unittest.mock import patch

DISABLE_ENV_PROXY = True  # 本机 DeepSeek 直连已验证；需要代理时设为 False
# Workflow 自动将控制台日志写入本次实验目录的 logs/console.log。
# 仅在本次运行临时移除代理环境变量，退出后恢复；不修改任何配置/密钥文件。
with patch.dict(os.environ):
    if DISABLE_ENV_PROXY:
        for name in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'http_proxy', 'https_proxy', 'all_proxy'):
            os.environ.pop(name, None)
    workflow_result = workflow.run(patient_id=PATIENT_ID, cutoff_day=CUTOFF_DAY, start_day=START_DAY)
log_path = Path(workflow_result['log_path'])
print({'status': workflow_result['status'], 'finished': workflow_result['finished'],
       'task_dir': workflow_result['task_dir'], 'log': str(log_path)})


In [ ]:
# 每个 Agent 调用是一个 turn；查看 Review 决策和重试原因，不展示完整患者病历。
turn_summary = [
    {'turn': turn['turn'], 'agent': turn['agent'], 'attempt': turn['attempt'],
     'status': turn['status'], 'reviews_turn': turn['reviews_turn'],
     'decision': (turn['output'] or {}).get('disposition'),
     'reason': (turn['output'] or {}).get('reason'), 'error': turn['error']}
    for turn in workflow_result['turns']
]
turn_summary
